# Generating COG mosaics <img align="right" src="../Supplementary_data/dea_logo.jpg">

* **[Sign up to the DEA Sandbox](https://app.sandbox.dea.ga.gov.au/)** to run this notebook interactively from a browser
* **Compatibility:** Notebook currently compatible with the `DEA Sandbox` environment
* **Products used:** 
[ga_ls_landcover_class_cyear_3](https://explorer.dea.ga.gov.au/ga_ls_landcover_class_cyear_3), [ga_ls5t_gm_cyear_3](https://explorer.dea.ga.gov.au/ga_ls5t_gm_cyear_3), [ga_ls7e_gm_cyear_3](https://explorer.dea.ga.gov.au/ga_ls7e_gm_cyear_3), [ga_ls8cls9c_gm_cyear_3](https://explorer.dea.ga.gov.au/ga_ls8cls9c_gm_cyear_3)

## Background

DEA products [are provided as tiles](https://knowledge.dea.ga.gov.au/guides/reference/collection_3_summary_grid/). The `datacube` library allows users to seamlessly work with these tiles and automatically mosaic them together for the area and time of interest. The data array loaded can then be exported as a standalone file.

An alternative method to generate mosaic files is by using `gdal` to join the tile files into a continental-scale Cloud-Optimised GeoTIFF (COG). COGs are an efficient format of GeoTIFF designed for cloud storage, which can be streamed and visualised more quickly in GIS software. COG data arrays are organised in square chunks, whereas standard GeoTIFFs are organised by bands. This means that, when visualising a region in GIS software, COGs load less data and are more efficient.

COGs also include a number of overviews (or pyramids) that simplify the view when zooming out. Overviews are generated by resampling the native data array into coarser resolutions. Different resampling algorithms can be applied depending on the type of data (e.g. categorical vs continuous), with the most common being `NEAREST`, `BILINEAR`, and `MODE`.

It is possible to define a colour scheme for categorical COGs, or to generate three-band colour composites for continuous COGs using a VRT (Virtual Raster). A VRT is essentially an XML file that contains the URLs or paths of the COGs, along with a colour scheme matching each raster value to a colour and label, or including three COGs as three bands for a colour composite (e.g. red, green, and blue bands for a true-colour visualisation).

However, VRTs are not just for applying colour schemes. They are a flexible tool that can be used for many other purposes (which are not explored in this notebook), such as:

* Combining multiple single-band COGs into a multiband raster
* Mosaicking multiple raster tiles into a seamless virtual layer

VRTs can then be streamed directly from GIS software such as QGIS or ESRI ArcGIS, providing access to the grid data with styling or band combinations applied (as shown in the example below, displaying the DEA Land Cover 2.0 for Tasmania and Victoria in QGIS).

![](../Supplementary_data/Example_VRTs/example-vrt-qgis.png)

## Description
This notebook explores how to perform the following actions:
* Generate mosaic Cloud-Optimised GeoTIFFs (COGs) from tiled satellite imagery
* Apply a DEA Land Cover colour scheme
* Create true-colour composites for the DEA Geomedian
  
It makes use of functions from the DEA Tools library, but it is also possible to run the analysis directly from the command line using the `make_cog_mosaic` and `make_styling_vrt` command line tools.
***

## Getting started

To generate mosaics, run the cells in the notebook starting with the "Load packages" cell.

### Load packages
Import the Python packages needed for this notebook.

In [1]:
from pathlib import Path

import sys
sys.path.insert(1, '../Tools/')

from dea_tools.mosaics.cog import make_cog_mosaic
from dea_tools.mosaics.vrt import make_styling_vrt

### Set output directory

Set the directory that will be used for the output mosaics.
Both local directories and locations on AWS S3 are supported; by default, this will be your current working directory.

In [2]:
output_dir = Path.cwd()

## Mosaics

### Land Cover
The following code cell is designed to run for every combination of Land Cover bands (`level3`, `level4`) and years included in input lists. For demonstration purposes, it is initially set to use only one band and one year, and it limits the mosaic to a few tiles.

In [3]:
# Inputs for the mosaic function
bands = ["level4"]
years = [2024]

# limit the mosaic to a few tiles
list_tiles = ["x46y47", "x46y48", "x46y49", "x46y50"]

In [4]:
for band in bands:
    for year in years:
        print(f"Running mosaic for band '{band}' and year '{year}'")

        make_cog_mosaic(
            product="ga_ls_landcover_class_cyear_3",
            band=band,
            time=year,
            freq="P1Y",
            version="2-0-0",
            dataset_maturity="final",
            product_dir="s3://dea-public-data/derivative/",  # DEA public AWS S3 bucket
            output_dir=output_dir,
            cog_blocksize=1024,
            overview_count=7,
            overview_resampling="MODE",  # for categorical data, mode is the best algorithm for the resampling
            compression_algo="ZSTD",
            compression_level=9,
            aws_unsigned=True,
            skip_existing=True,  # Set False if want to overwrite existing files
            list_tiles=list_tiles,  # set it to None if need a national mosaic
        )

2025-07-30 02:18:39,102 - dea_tools.mosaics.cog - INFO - [ga_ls_landcover_class_cyear_3] [2-0-0] [2024] [level4]: Using parameters {'product': 'ga_ls_landcover_class_cyear_3', 'band': 'level4', 'time': 2024, 'freq': 'P1Y', 'version': '2-0-0', 'dataset_maturity': 'final', 'product_dir': 's3://dea-public-data/derivative/', 'output_dir': PosixPath('/home/jovyan/dev/How_to_guides'), 'cog_blocksize': 1024, 'overview_count': 7, 'overview_resampling': 'MODE', 'compression_algo': 'ZSTD', 'compression_level': 9, 'aws_unsigned': True, 'skip_existing': True, 'list_tiles': ['x46y47', 'x46y48', 'x46y49', 'x46y50'], 'vsi_method': 'vsicurl', 'log': <Logger dea_tools.mosaics.cog (INFO)>}
2025-07-30 02:18:39,102 - dea_tools.mosaics.cog - INFO - [ga_ls_landcover_class_cyear_3] [2-0-0] [2024] [level4]: Using input data product directory: dea-public-data/derivative
2025-07-30 02:18:39,102 - dea_tools.mosaics.cog - INFO - [ga_ls_landcover_class_cyear_3] [2-0-0] [2024] [level4] Output path: /home/jovyan/dev

Running mosaic for band 'level4' and year '2024'


2025-07-30 02:19:31,400 - dea_tools.mosaics.cog - INFO - [ga_ls_landcover_class_cyear_3] [2-0-0] [2024] [level4]: Number of tiles to mosaic: 4
2025-07-30 02:19:31,401 - dea_tools.mosaics.cog - INFO - [ga_ls_landcover_class_cyear_3] [2-0-0] [2024] [level4]: Writing data to temporary folder: /tmp/tmp3k1fpko7
2025-07-30 02:19:31,402 - dea_tools.mosaics.cog - INFO - [ga_ls_landcover_class_cyear_3] [2-0-0] [2024] [level4]: Building virtual raster (VRT)
2025-07-30 02:19:31,523 - dea_tools.mosaics.cog - INFO - [ga_ls_landcover_class_cyear_3] [2-0-0] [2024] [level4]: Converting VRT to COG mosaic


0...10...20...30...40...50...60...70...80...90...100 - done.
Input file size is 3200, 12800
0...10...20...30...40...50...60...70...80...90...100 - done.


2025-07-30 02:19:32,689 - dea_tools.mosaics.cog - INFO - [ga_ls_landcover_class_cyear_3] [2-0-0] [2024] [level4]: Writing data locally: /home/jovyan/dev/How_to_guides/ga_ls_landcover_class_cyear_3/2-0-0/continental_mosaics/2024--P1Y/ga_ls_landcover_class_cyear_3_mosaic_2024--P1Y_level4.tif


Running the same analysis via the command line would be the following.

In [5]:
bands = ["level4"]
years = [2024]
list_tiles = ["x46y47", "x46y48", "x46y49", "x46y50"]

for band in bands:
    for year in years:
        !make_cog_mosaic \
            --product ga_ls_landcover_class_cyear_3 \
            --band {band} \
            --time {year} \
            --freq P1Y \
            --version 2-0-0 \
            --dataset_maturity final \
            --product_dir s3://dea-public-data/derivative/ \
            --output_dir {output_dir} \
            --cog_blocksize 1024 \
            --overview_count 7 \
            --overview_resampling MODE \
            --compression_algo ZSTD \
            --compression_level 9 \
            --aws_unsigned \
            --skip_existing \
            --list_tiles {','.join(list_tiles)}


2025-07-30 02:19:34,712 - dea_tools.mosaics.cog - INFO - [ga_ls_landcover_class_cyear_3] [2-0-0] [2024] [level4]: Using parameters {'product': 'ga_ls_landcover_class_cyear_3', 'band': 'level4', 'time': '2024', 'freq': 'P1Y', 'version': '2-0-0', 'dataset_maturity': 'final', 'product_dir': 's3://dea-public-data/derivative/', 'output_dir': '/home/jovyan/dev/How_to_guides', 'cog_blocksize': 1024, 'overview_count': 7, 'overview_resampling': 'MODE', 'compression_algo': 'ZSTD', 'compression_level': 9, 'aws_unsigned': True, 'skip_existing': True, 'list_tiles': ['x46y47', 'x46y48', 'x46y49', 'x46y50'], 'vsi_method': 'vsicurl', 'log': <Logger dea_tools.mosaics.cog (INFO)>}
2025-07-30 02:19:34,712 - dea_tools.mosaics.cog - INFO - [ga_ls_landcover_class_cyear_3] [2-0-0] [2024] [level4]: Using input data product directory: dea-public-data/derivative
2025-07-30 02:19:34,712 - dea_tools.mosaics.cog - INFO - [ga_ls_landcover_class_cyear_3] [2-0-0] [2024] [level4] Output path: /home/jovyan/dev/How_to_g

### Geomedian
DEA Geomedian products are split according to the sensor used for the input data (Landsat 5, Landsat 7, or Landsat 8 and 9). The following code cell is designed to generate mosaics for different sets of years for each product, in combination with a list of bands. Currently, RGB bands are mosaicked and will later be used to generate a true-colour VRT.

For demonstration purposes, the example below will mosaic Landsat 8 and 9 Geomedians only. To mosaic Landsat 5 and Landsat 7, uncomment the relevant lines in `product_years` and re-run the cell below.

In [6]:
# dictionary for defining the years of interest for each geomedian product
products_years = {
    # "ga_ls5t_gm_cyear_3": [1990],
    # "ga_ls7e_gm_cyear_3": [2016],
    "ga_ls8cls9c_gm_cyear_3": [2024],
}

# bands for RGB composites
bands = ["nbart_red", "nbart_green", "nbart_blue"]

# limit the mosaic to a few tiles
list_tiles = ["x46y47", "x46y48", "x46y49", "x46y50"]

In [7]:
for product, years in products_years.items():
    for year in years:
        for band in bands:
            make_cog_mosaic(
                product=product,
                band=band,
                time=str(year),
                freq="P1Y",
                version="4-0-0",
                dataset_maturity="final",
                product_dir="s3://dea-public-data/derivative/",  # DEA public AWS S3 bucket
                output_dir=output_dir,  # use the current directory
                cog_blocksize=1024,
                overview_count=7,
                overview_resampling="BILINEAR",  # for continuous data, bilinear is usually the best choice for the resampling algorithm
                compression_algo="ZSTD",
                compression_level=9,
                aws_unsigned=True,
                skip_existing=True,  # Set False if want to overwrite existing files
                list_tiles=list_tiles,  # set it to None if need a national mosaic
            )

2025-07-30 02:19:35,053 - dea_tools.mosaics.cog - INFO - [ga_ls8cls9c_gm_cyear_3] [4-0-0] [2024] [nbart_red]: Using parameters {'product': 'ga_ls8cls9c_gm_cyear_3', 'band': 'nbart_red', 'time': '2024', 'freq': 'P1Y', 'version': '4-0-0', 'dataset_maturity': 'final', 'product_dir': 's3://dea-public-data/derivative/', 'output_dir': PosixPath('/home/jovyan/dev/How_to_guides'), 'cog_blocksize': 1024, 'overview_count': 7, 'overview_resampling': 'BILINEAR', 'compression_algo': 'ZSTD', 'compression_level': 9, 'aws_unsigned': True, 'skip_existing': True, 'list_tiles': ['x46y47', 'x46y48', 'x46y49', 'x46y50'], 'vsi_method': 'vsicurl', 'log': <Logger dea_tools.mosaics.cog (INFO)>}
2025-07-30 02:19:35,054 - dea_tools.mosaics.cog - INFO - [ga_ls8cls9c_gm_cyear_3] [4-0-0] [2024] [nbart_red]: Using input data product directory: dea-public-data/derivative
2025-07-30 02:19:35,054 - dea_tools.mosaics.cog - INFO - [ga_ls8cls9c_gm_cyear_3] [4-0-0] [2024] [nbart_red] Output path: /home/jovyan/dev/How_to_gu

0...10...20...30...40...50...60...70...80...90...100 - done.
Input file size is 3200, 12800
0...10...20...30...40...50...60...70...80...90...

2025-07-30 02:20:15,808 - dea_tools.mosaics.cog - INFO - [ga_ls8cls9c_gm_cyear_3] [4-0-0] [2024] [nbart_red]: Writing data locally: /home/jovyan/dev/How_to_guides/ga_ls8cls9c_gm_cyear_3/4-0-0/continental_mosaics/2024--P1Y/ga_ls8cls9c_gm_cyear_3_mosaic_2024--P1Y_nbart_red.tif
2025-07-30 02:20:15,856 - dea_tools.mosaics.cog - INFO - [ga_ls8cls9c_gm_cyear_3] [4-0-0] [2024] [nbart_green]: Using parameters {'product': 'ga_ls8cls9c_gm_cyear_3', 'band': 'nbart_green', 'time': '2024', 'freq': 'P1Y', 'version': '4-0-0', 'dataset_maturity': 'final', 'product_dir': 's3://dea-public-data/derivative/', 'output_dir': PosixPath('/home/jovyan/dev/How_to_guides'), 'cog_blocksize': 1024, 'overview_count': 7, 'overview_resampling': 'BILINEAR', 'compression_algo': 'ZSTD', 'compression_level': 9, 'aws_unsigned': True, 'skip_existing': True, 'list_tiles': ['x46y47', 'x46y48', 'x46y49', 'x46y50'], 'vsi_method': 'vsicurl', 'log': <Logger dea_tools.mosaics.cog (INFO)>}
2025-07-30 02:20:15,857 - dea_tools.mosai

100 - done.


2025-07-30 02:20:52,541 - dea_tools.mosaics.cog - INFO - [ga_ls8cls9c_gm_cyear_3] [4-0-0] [2024] [nbart_green]: Number of tiles to mosaic: 4
2025-07-30 02:20:52,542 - dea_tools.mosaics.cog - INFO - [ga_ls8cls9c_gm_cyear_3] [4-0-0] [2024] [nbart_green]: Writing data to temporary folder: /tmp/tmpoyqt9423
2025-07-30 02:20:52,543 - dea_tools.mosaics.cog - INFO - [ga_ls8cls9c_gm_cyear_3] [4-0-0] [2024] [nbart_green]: Building virtual raster (VRT)
2025-07-30 02:20:52,669 - dea_tools.mosaics.cog - INFO - [ga_ls8cls9c_gm_cyear_3] [4-0-0] [2024] [nbart_green]: Converting VRT to COG mosaic


...

100 - done.


2025-07-30 02:21:32,346 - dea_tools.mosaics.cog - INFO - [ga_ls8cls9c_gm_cyear_3] [4-0-0] [2024] [nbart_blue]: Number of tiles to mosaic: 4
2025-07-30 02:21:32,348 - dea_tools.mosaics.cog - INFO - [ga_ls8cls9c_gm_cyear_3] [4-0-0] [2024] [nbart_blue]: Writing data to temporary folder: /tmp/tmp1lcf61yb
2025-07-30 02:21:32,349 - dea_tools.mosaics.cog - INFO - [ga_ls8cls9c_gm_cyear_3] [4-0-0] [2024] [nbart_blue]: Building virtual raster (VRT)
2025-07-30 02:21:32,468 - dea_tools.mosaics.cog - INFO - [ga_ls8cls9c_gm_cyear_3] [4-0-0] [2024] [nbart_blue]: Converting VRT to COG mosaic


0...10...20...30...40...50...60...70...80...90...100 - done.
Input file size is 3200, 12800
0...10...20...30...40...50...60...70...80...90...

2025-07-30 02:21:36,873 - dea_tools.mosaics.cog - INFO - [ga_ls8cls9c_gm_cyear_3] [4-0-0] [2024] [nbart_blue]: Writing data locally: /home/jovyan/dev/How_to_guides/ga_ls8cls9c_gm_cyear_3/4-0-0/continental_mosaics/2024--P1Y/ga_ls8cls9c_gm_cyear_3_mosaic_2024--P1Y_nbart_blue.tif


100 - done.


## Generate VRTs to add colour scheme to mosaics

### Categorical data - Land Cover
The following code cells generate VRTs that apply a predefined colour scheme to the detailed DEA Land Cover 2.0 classification (`level4`). This enables direct visualisation of the grid data using the official colours, consistent with other DEA Land Cover notebooks (e.g. [DEA_Land_Cover.ipynb](../DEA_products/DEA_Land_Cover.ipynb)) and the [DEA Maps](https://maps.dea.ga.gov.au/#share=s-y3DVNKDJvUmJm18IrojvvWWXOmo).

`make_styling_vrt` also accepts a `style_json_path` input parameter that allows using a custom colour scheme. Run `help(make_styling_vrt)` in a code cell for additional instructions.

The code can be easily adjusted to process multiple Land Cover bands (detailed `level4` and basic `level3`) and/or multiple years. Simply add the desired values to the list variables below (`bands` and `years`).

It is assumed that the corresponding mosaics already exist, and the extent of the VRTs will be limited to that of the mosaics. For example, if the mosaic is composed of four tiles, the VRT will include only those four tiles.

In [8]:
# inputs for the VRT function
bands = ["level4"]
years = [2024]

In [9]:
for band in bands:
    for year in years:
        make_styling_vrt(
            product="ga_ls_landcover_class_cyear_3",
            version="2-0-0",
            time=year,
            freq="P1Y",
            cog_dir=output_dir,  # we'll use the same directory used to generate the mosaics
            output_dir=output_dir,
            band=band,
        )

2025-07-30 02:21:36,927 - dea_tools.mosaics.vrt - INFO - Creating colour VRTs for Land Cover [ga_ls_landcover_class_cyear_3] [2-0-0] [2024] [level4]: Using parameters {'product': 'ga_ls_landcover_class_cyear_3', 'version': '2-0-0', 'band': 'level4', 'time': 2024, 'freq': 'P1Y', 'cog_dir': '/home/jovyan/dev/How_to_guides', 'output_dir': '/home/jovyan/dev/How_to_guides', 'style_json_path': None, 'log': <Logger dea_tools.mosaics.vrt (INFO)>}
2025-07-30 02:21:36,928 - dea_tools.mosaics.vrt - INFO - [ga_ls_landcover_class_cyear_3] [2-0-0] [2024] [level4]: Identifying input data from local file system: /home/jovyan/dev/How_to_guides/ga_ls_landcover_class_cyear_3/2-0-0/continental_mosaics/2024--P1Y/ga_ls_landcover_class_cyear_3_mosaic_2024--P1Y_level4.tif
2025-07-30 02:21:36,929 - dea_tools.mosaics.vrt - INFO - [ga_ls_landcover_class_cyear_3] [2-0-0] [2024] [level4] - Output path: /home/jovyan/dev/How_to_guides/ga_ls_landcover_class_cyear_3/2-0-0/continental_mosaics/2024--P1Y/ga_ls_landcover_

0...10...20...30...40...50...60...70...80...90...100 - done.


### Three-bands composites
The following code generates three-band composites from any three existing band mosaics. It is possible to customise the composite by changing the `r_channel`, `g_channel`, and `b_channel` variables.

The code can be run for multiple years and products by adjusting the `product_years` dictionary.

In [10]:
products_years = {
    # 'ga_ls5t_gm_cyear_3': [1990],
    # 'ga_ls7e_gm_cyear_3': [2016],
    "ga_ls8cls9c_gm_cyear_3": [2024]
}

r_channel = "nbart_red"
g_channel = "nbart_green"
b_channel = "nbart_blue"

for product, years in products_years.items():
    for year in years:

        make_styling_vrt(
            product=product,
            version="4-0-0",
            time=year,
            freq="P1Y",
            cog_dir=output_dir,  # we'll use the same directory used to generate the mosaics
            output_dir=output_dir,
            r_channel_band=r_channel,
            g_channel_band=g_channel,
            b_channel_band=b_channel,
        )

2025-07-30 02:21:36,998 - dea_tools.mosaics.vrt - INFO - Creating colour VRTs for Geomedian [ga_ls8cls9c_gm_cyear_3] [4-0-0] [2024] [nbart_red] [nbart_green] [nbart_blue]: Using parameters {'product': 'ga_ls8cls9c_gm_cyear_3', 'version': '4-0-0', 'time': 2024, 'freq': 'P1Y', 'cog_dir': '/home/jovyan/dev/How_to_guides', 'output_dir': '/home/jovyan/dev/How_to_guides', 'r_channel_band': 'nbart_red', 'g_channel_band': 'nbart_green', 'b_channel_band': 'nbart_blue', 'vrt_name': None, 'log': <Logger dea_tools.mosaics.vrt (INFO)>}
2025-07-30 02:21:36,999 - dea_tools.mosaics.vrt - INFO - [ga_ls8cls9c_gm_cyear_3] [4-0-0] [2024] [nbart_red] [nbart_green] [nbart_blue]: Identifying input data from local file system:
-/home/jovyan/dev/How_to_guides/ga_ls8cls9c_gm_cyear_3/4-0-0/continental_mosaics/2024--P1Y/ga_ls8cls9c_gm_cyear_3_mosaic_2024--P1Y_nbart_red.tif
-/home/jovyan/dev/How_to_guides/ga_ls8cls9c_gm_cyear_3/4-0-0/continental_mosaics/2024--P1Y/ga_ls8cls9c_gm_cyear_3_mosaic_2024--P1Y_nbart_green

0...10...20...30...40...50...60...70...80...90...100 - done.


## Delete output folders and their content 
Once the output data is no longer needed, it may be convenient to delete the newly created folders and their contents using a command-line instruction. This is because the DEA Sandbox does not permit the deletion of non-empty folders without first removing the files they contain.

In [11]:
!rm -rf {output_dir}/ga_ls_landcover_class_cyear_3 {output_dir}/ga_ls5t_gm_cyear_3 {output_dir}/ga_ls7e_gm_cyear_3 {output_dir}/ga_ls8cls9c_gm_cyear_3

***

## Additional information

**License:** The code in this notebook is licensed under the [Apache License, Version 2.0](https://www.apache.org/licenses/LICENSE-2.0). 
Digital Earth Australia data is licensed under the [Creative Commons by Attribution 4.0](https://creativecommons.org/licenses/by/4.0/) license.

**Contact:** If you need assistance, please post a question on the [Open Data Cube Discord chat](https://discord.com/invite/4hhBQVas5U) or on the [GIS Stack Exchange](https://gis.stackexchange.com/questions/ask?tags=open-data-cube) using the `open-data-cube` tag (you can view previously asked questions [here](https://gis.stackexchange.com/questions/tagged/open-data-cube)).
If you would like to report an issue with this notebook, you can file one on [GitHub](https://github.com/GeoscienceAustralia/dea-notebooks).

**Last modified:** July 2025

## Tags
<!-- Browse all available tags on the DEA User Guide's [Tags Index](https://knowledge.dea.ga.gov.au/genindex/) -->